# Singapore Jobs Analytics — Data Handling Notebook

**Module 1 Assignment · NTU DSAI**

---

## 1. Business Case

| | |
|---|---|
| **Scenario** | Talent acquisition teams and recruitment agencies in Singapore need to prioritise roles and calibrate offers in a competitive market. |
| **Objective** | Identify the most in-demand job roles, competitive salary benchmarks, fastest-growing industries, and hiring seasonality using ~1M job postings from MyCareersFuture.sg. |
| **Target users** | Recruiters and hiring managers who need to answer: *Which roles should I source for? What salary should I offer? Is this industry growing?* |
| **Value** | Cuts research time from days to minutes; grounds compensation decisions in live market data rather than gut feel. |


---
## 2. Setup & Data Loading


In [1]:
import plotly.io as pio; pio.renderers.default = 'notebook_connected'

In [2]:
import duckdb
import pandas as pd
import numpy as np
import json
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Libraries loaded.')

Libraries loaded.


In [ ]:
# SGJobData_copy.duckdb avoids locking conflicts if the original is open in another kernel.
# Switch to 'SGJobData.duckdb' with read_only=True once the other kernel is closed.
import os
# DB_FILE = 'SGJobData_copy.duckdb' if os.path.exists('SGJobData_copy.duckdb') else 'SGJobData.duckdb'
DB_FILE = 'SGJobData.duckdb'
con = duckdb.connect(DB_FILE, read_only=True)

tables = con.execute('SHOW TABLES').df()
row_count = con.execute('SELECT COUNT(*) FROM jobs').fetchone()[0]
print(f'Connected to: {DB_FILE}')
print(f'Tables: {tables["name"].tolist()}')
print(f'Row count: {row_count:,}')

Connected to: SGJobData.duckdb
Tables: ['jobs']
Row count: 1,048,585


In [4]:
# Load full table into a DataFrame (~350 MB in memory — expect 10–30 s on first run)
df_raw = con.execute('SELECT * FROM jobs').df()
print(f'Shape: {df_raw.shape}')
df_raw.dtypes

Shape: (1048585, 22)


categories                                       str
employmentTypes                                  str
metadata_expiryDate                   datetime64[us]
metadata_isPostedOnBehalf                       bool
metadata_jobPostId                               str
metadata_newPostingDate               datetime64[us]
metadata_originalPostingDate          datetime64[us]
metadata_repostCount                           int64
metadata_totalNumberJobApplication             int64
metadata_totalNumberOfView                     int64
minimumYearsExperience                         int64
numberOfVacancies                              int64
occupationId                                  object
positionLevels                                   str
postedCompany_name                               str
salary_maximum                                 int64
salary_minimum                                 int64
salary_type                                      str
status_id                                     

In [5]:
df_raw.head(3)

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,minimumYearsExperience,numberOfVacancies,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
0,"[{""id"":13,""category"":""Environment / Health""},{...",Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,0,1,None,Executive,WORKSTONE PTE. LTD.,2800,2000,Monthly,0,Closed,Food Technologist - Clementi | Entry Level | U...,"2,400.00"
1,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273977,2023-04-08,2023-04-08,0,0,55,2,2,None,Executive,TRUST RECRUIT PTE. LTD.,5500,4000,Monthly,0,Closed,"Software Engineer (Fab Support) (Java, CIM, Up...","4,750.00"
2,"[{""id"":33,""category"":""Repair and Maintenance""}]",Full Time,2023-04-22,False,MCF-2023-0273994,2023-04-08,2023-04-08,0,7,99,3,1,None,Senior Executive,PU TIEN SERVICES PTE. LTD.,4600,3800,Monthly,0,Closed,Senior Technician,"4,200.00"


---
## 3. Data Quality Assessment


In [6]:
# Missing values per column
null_counts = df_raw.isna().sum()
null_pct = (null_counts / len(df_raw) * 100).round(2)
null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_summary[null_summary['null_count'] > 0].sort_values('null_count', ascending=False)

,null_count,null_pct
occupationId,1048585,100.00
categories,3988,0.38
metadata_expiryDate,3988,0.38
employmentTypes,3988,0.38
metadata_jobPostId,3988,0.38
metadata_newPostingDate,3988,0.38
metadata_originalPostingDate,3988,0.38
positionLevels,3988,0.38
postedCompany_name,3988,0.38
salary_type,3988,0.38


**Observations:**
- `occupationId` is **100% null** → drop it.
- ~3,988 rows have `title`, `categories`, `positionLevels`, etc. all null — these are empty placeholder rows.
- 3,987 of those nulls are **exact duplicates** of a single empty record → remove.


In [7]:
n_dupes = df_raw.duplicated().sum()
print(f'Duplicate rows: {n_dupes:,}')
print(f'All duplicates have null title: {df_raw[df_raw.duplicated()]["title"].isna().all()}')

Duplicate rows: 3,987
All duplicates have null title: True


In [8]:
# Salary distribution — only rows with title present (non-null rows)
sal = df_raw.dropna(subset=['title'])['average_salary']

print('Salary (average_salary) percentiles — non-null rows:')
print(sal.describe(percentiles=[.05, .25, .5, .75, .95, .99]))

print(f'\nRows with average_salary > 50,000 (SGD/month): {(sal > 50000).sum():,}')
print(f'Rows with average_salary == 0:                  {(sal == 0).sum():,}')

Salary (average_salary) percentiles — non-null rows:
count    1,044,597.00
mean         4,787.65
std         25,524.97
min              1.00
5%           1,950.00
25%          2,900.00
50%          3,800.00
75%          5,500.00
95%         10,500.00
99%         16,750.00
max     12,666,400.00
Name: average_salary, dtype: float64

Rows with average_salary > 50,000 (SGD/month): 405
Rows with average_salary == 0:                  0


**Salary anomalies detected:**
- All salary values are labelled `salary_type = Monthly` (SGD).
- The median is ~S$3,800/month and 99th percentile ~S$16,750/month — both realistic for Singapore.
- However, ~840 rows show `average_salary > S$50,000/month` (e.g., S$25 million/month) — these are clearly erroneous `salary_maximum` entries in the source data.
- **Cleaning rule:** keep rows where `500 ≤ average_salary ≤ 50,000`.


In [9]:
# Categories are stored as a JSON array of {id, category} objects
sample_cats = df_raw['categories'].dropna().head(5).tolist()
for s in sample_cats:
    print(json.loads(s))

[{'id': 13, 'category': 'Environment / Health'}, {'id': 25, 'category': 'Manufacturing'}, {'id': 36, 'category': 'Sciences / Laboratory / R&D'}]
[{'id': 21, 'category': 'Information Technology'}]
[{'id': 33, 'category': 'Repair and Maintenance'}]
[{'id': 21, 'category': 'Information Technology'}]
[{'id': 2, 'category': 'Admin / Secretarial'}]


In [10]:
print('--- employmentTypes ---')
print(df_raw['employmentTypes'].value_counts(dropna=False).to_string())

print('\n--- positionLevels ---')
print(df_raw['positionLevels'].value_counts(dropna=False).to_string())

--- employmentTypes ---
employmentTypes
Permanent                458139
Full Time                393352
Contract                 139182
Part Time                 25431
Temporary                 18241
Internship/Attachment      6959
NaN                        3988
Freelance                  2139
Flexi-work                 1154

--- positionLevels ---
positionLevels
Executive            253701
Junior Executive     167656
Non-executive        131608
Fresh/entry level    118661
Professional         112208
Manager              110122
Senior Executive     100459
Middle Management     27375
Senior Management     22807
NaN                    3988


In [11]:
dates = df_raw['metadata_originalPostingDate'].dropna()
print(f'Date range: {dates.min().date()} → {dates.max().date()}')
print(f'Span: ~{(dates.max() - dates.min()).days // 30} months')

Date range: 2022-10-03 → 2024-05-29
Span: ~20 months


---
## 4. Data Cleaning


In [12]:
# Step 1 — drop rows with no job title (also removes the 3,988 empty/duplicate rows)
df = df_raw.dropna(subset=['title']).drop_duplicates().reset_index(drop=True)
print(f'After removing nulls & duplicates: {len(df):,} rows  (was {len(df_raw):,})')

After removing nulls & duplicates: 1,044,597 rows  (was 1,048,585)


In [13]:
# occupationId is entirely null — drop it.
# Also drop helper IDs and redundant columns not needed for analytics.
DROP_COLS = [
    'occupationId', 'status_id', 'metadata_jobPostId',
    'metadata_isPostedOnBehalf', 'metadata_expiryDate',
    'metadata_newPostingDate',
]
df.drop(columns=DROP_COLS, inplace=True)
print('Remaining columns:', df.columns.tolist())

Remaining columns: ['categories', 'employmentTypes', 'metadata_originalPostingDate', 'metadata_repostCount', 'metadata_totalNumberJobApplication', 'metadata_totalNumberOfView', 'minimumYearsExperience', 'numberOfVacancies', 'positionLevels', 'postedCompany_name', 'salary_maximum', 'salary_minimum', 'salary_type', 'status_jobStatus', 'title', 'average_salary']


In [14]:
# Step 2 — remove salary outliers
before = len(df)
df = df[(df['average_salary'] >= 500) & (df['average_salary'] <= 50_000)].reset_index(drop=True)
print(f'Removed {before - len(df):,} rows with implausible salaries.')
print(f'Clean dataset: {len(df):,} rows')

Removed 7,853 rows with implausible salaries.
Clean dataset: 1,036,744 rows


In [15]:
# Step 3 — parse JSON categories → extract first (primary) category as a plain string
def get_primary_category(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return 'Other'
    try:
        cats = json.loads(s)
        if cats:
            return cats[0]['category']
    except Exception:
        pass
    return 'Other'

df['primary_category'] = df['categories'].apply(get_primary_category)
print('Top 15 primary categories:')
print(df['primary_category'].value_counts().head(15).to_string())

Top 15 primary categories:
primary_category
Admin / Secretarial                 101686
Information Technology               99937
Engineering                          99552
Accounting / Auditing / Taxation     78237
Building and Construction            73871
Customer Service                     63983
F&B                                  58899
Banking and Finance                  46382
Logistics / Supply Chain             44215
Sales / Retail                       36843
Education and Training               35034
Healthcare / Pharmaceutical          32672
Others                               24368
Consulting                           24085
Marketing / Public Relations         23141


In [16]:
# Step 4 — normalise job title (lowercase + strip)
df['title_norm'] = df['title'].str.lower().str.strip()

print('Top 10 normalised titles (before vs after):')
pd.DataFrame({
    'original': df['title'].value_counts().head(5).index.tolist(),
    'normalised': df['title_norm'].value_counts().head(5).index.tolist(),
})

Top 10 normalised titles (before vs after):


,original,normalised
0,SUPERVISOR,supervisor
1,Supervisor,chef
2,Quantity Surveyor,quantity surveyor
3,Accounts Executive,project manager
4,CHEF,accounts executive


---
## 5. Feature Engineering


In [17]:
# posting_date, posting_month (YYYY-MM string), posting_year
df['posting_date'] = pd.to_datetime(df['metadata_originalPostingDate'])
df['posting_month'] = df['posting_date'].dt.to_period('M').astype(str)
df['posting_year'] = df['posting_date'].dt.year

# is_repost flag
df['is_repost'] = df['metadata_repostCount'] > 0

print(df[['posting_date', 'posting_month', 'posting_year', 'is_repost']].head(3))
print(f"\nRepost rate: {df['is_repost'].mean()*100:.1f}%")

  posting_date posting_month  posting_year  is_repost
0   2023-03-30       2023-03          2023       True
1   2023-04-08       2023-04          2023      False
2   2023-04-08       2023-04          2023      False

Repost rate: 4.1%


In [18]:
# Map positionLevels → seniority band (simpler 4-tier hierarchy)
SENIORITY_MAP = {
    'Fresh/entry level': 'Junior',
    'Junior Executive':  'Junior',
    'Non-executive':     'Mid',
    'Executive':         'Mid',
    'Professional':      'Mid',
    'Senior Executive':  'Senior',
    'Manager':           'Senior',
    'Middle Management': 'Management',
    'Senior Management': 'Management',
}
df['seniority'] = df['positionLevels'].map(SENIORITY_MAP).fillna('Other')

print('Seniority distribution:')
print(df['seniority'].value_counts().to_string())

Seniority distribution:
seniority
Mid           494224
Junior        282840
Senior        209770
Management     49910


In [19]:
# Salary bands aligned to Singapore market tiers
BAND_ORDER = [
    'Entry (<$2.5k)',
    'Junior ($2.5k-$4.5k)',
    'Mid ($4.5k-$7.5k)',
    'Senior ($7.5k-$12k)',
    'Premium (>$12k)',
]

def salary_band(sal):
    if sal < 2500:
        return 'Entry (<$2.5k)'
    elif sal < 4500:
        return 'Junior ($2.5k-$4.5k)'
    elif sal < 7500:
        return 'Mid ($4.5k-$7.5k)'
    elif sal < 12000:
        return 'Senior ($7.5k-$12k)'
    return 'Premium (>$12k)'

df['salary_band'] = df['average_salary'].apply(salary_band)

print('Salary band distribution:')
print(df['salary_band'].value_counts()[BAND_ORDER].to_string())

Salary band distribution:
salary_band
Entry (<$2.5k)          140822
Junior ($2.5k-$4.5k)    511939
Mid ($4.5k-$7.5k)       239278
Senior ($7.5k-$12k)     110587
Premium (>$12k)          34118


In [20]:
# Final feature summary
print(f'Clean dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.describe(include='all').T[['count', 'unique', 'top', 'freq', 'mean', 'std', 'min', '50%', 'max']]

Clean dataset shape: (1036744, 24)
Columns: ['categories', 'employmentTypes', 'metadata_originalPostingDate', 'metadata_repostCount', 'metadata_totalNumberJobApplication', 'metadata_totalNumberOfView', 'minimumYearsExperience', 'numberOfVacancies', 'positionLevels', 'postedCompany_name', 'salary_maximum', 'salary_minimum', 'salary_type', 'status_jobStatus', 'title', 'average_salary', 'primary_category', 'title_norm', 'posting_date', 'posting_month', 'posting_year', 'is_repost', 'seniority', 'salary_band']


,count,unique,top,freq,mean,std,min,50%,max
categories,1036744,21022,"[{""id"":21,""category"":""Information Technology""}]",92686,NaN,NaN,NaN,NaN,NaN
employmentTypes,1036744,8,Permanent,456977,NaN,NaN,NaN,NaN,NaN
metadata_originalPostingDate,1036744,NaN,NaN,NaN,2023-11-01 02:15:57.523941,NaN,2022-10-03 00:00:00,2023-10-30 00:00:00,2024-05-29 00:00:00
metadata_repostCount,"1,036,744.00",NaN,NaN,NaN,0.05,0.28,0.00,0.00,2.00
metadata_totalNumberJobApplication,"1,036,744.00",NaN,NaN,NaN,2.14,10.36,0.00,0.00,"1,342.00"
metadata_totalNumberOfView,"1,036,744.00",NaN,NaN,NaN,26.74,81.51,0.00,4.00,"8,190.00"
minimumYearsExperience,"1,036,744.00",NaN,NaN,NaN,2.80,2.53,0.00,2.00,88.00
numberOfVacancies,"1,036,744.00",NaN,NaN,NaN,2.66,10.62,1.00,1.00,999.00
positionLevels,1036744,9,Executive,252857,NaN,NaN,NaN,NaN,NaN
postedCompany_name,1036744,52918,THE SUPREME HR ADVISORY PTE. LTD.,61581,NaN,NaN,NaN,NaN,NaN


---
## 6. Exploratory Data Analysis (EDA)

All charts below use Plotly and are interactive in-notebook.


In [21]:
# ── Monthly posting trend ────────────────────────────────────────────────────
monthly = df.groupby('posting_month').size().reset_index(name='count')

fig = px.area(
    monthly, x='posting_month', y='count',
    title='Monthly Job Postings — Oct 2022 to May 2024',
    labels={'posting_month': 'Month', 'count': 'Postings'},
    color_discrete_sequence=['#003DA5'],
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**Finding:** Posting volume was negligible at end of 2022, ramped sharply from Feb 2023, and stabilised at ~70–85k/month from mid-2023 onward — suggesting the dataset's coverage began in earnest in early 2023.


In [22]:
# ── Top industries ───────────────────────────────────────────────────────────
ind_c = df['primary_category'].value_counts().head(15).reset_index()
ind_c.columns = ['industry', 'count']

fig = px.bar(
    ind_c, x='count', y='industry', orientation='h',
    title='Top 15 Industries by Job Posting Volume',
    labels={'count': 'Postings', 'industry': ''},
    color='count', color_continuous_scale='Blues',
)
fig.update_layout(
    height=520,
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
)
fig.show()

In [23]:
# ── Top job titles (normalised) ───────────────────────────────────────────────
tc = df['title_norm'].value_counts().head(20).reset_index()
tc.columns = ['title', 'count']

fig = px.bar(
    tc, x='count', y='title', orientation='h',
    title='Top 20 Job Titles (normalised)',
    labels={'count': 'Postings', 'title': ''},
    color='count', color_continuous_scale='Greens',
)
fig.update_layout(
    height=560,
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
)
fig.show()

**Finding:** Title normalisation (lowercase) consolidates 'SUPERVISOR', 'Supervisor', 'supervisor' etc.  
Top roles — `supervisor`, `quantity surveyor`, `accounts executive`, `project manager` — reflect Singapore's construction boom and strong administrative demand.


In [24]:
# ── Median salary by position level ──────────────────────────────────────────
LEVEL_ORDER = [
    'Fresh/entry level', 'Junior Executive', 'Non-executive',
    'Executive', 'Professional', 'Senior Executive',
    'Manager', 'Middle Management', 'Senior Management',
]
avail = [l for l in LEVEL_ORDER if l in df['positionLevels'].values]
sal_lvl = (
    df.groupby('positionLevels')['average_salary']
    .median()
    .reindex(avail)
    .reset_index()
)
sal_lvl.columns = ['level', 'median_salary']

fig = px.bar(
    sal_lvl, x='level', y='median_salary',
    title='Median Monthly Salary by Position Level (SGD)',
    labels={'median_salary': 'Median Salary (SGD/month)', 'level': ''},
    color='median_salary', color_continuous_scale='RdYlGn',
)
fig.update_layout(xaxis_tickangle=-25, coloraxis_showscale=False)
fig.show()

In [25]:
# ── Employment type distribution ─────────────────────────────────────────────
emp_c = df['employmentTypes'].value_counts().reset_index()
emp_c.columns = ['type', 'count']

fig = px.pie(
    emp_c, values='count', names='type',
    title='Employment Type Distribution',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

In [26]:
# ── Salary band distribution ──────────────────────────────────────────────────
band_c = df['salary_band'].value_counts().reindex(BAND_ORDER).reset_index()
band_c.columns = ['band', 'count']

fig = px.bar(
    band_c, x='band', y='count',
    title='Postings by Salary Band',
    labels={'band': 'Salary Band', 'count': 'Postings'},
    color='band',
    color_discrete_sequence=px.colors.sequential.Blues[1:],
)
fig.update_layout(xaxis_tickangle=-15, showlegend=False)
fig.show()

In [27]:
# ── Monthly trend for top 5 industries ────────────────────────────────────────
top5 = df['primary_category'].value_counts().head(5).index.tolist()
ind_mth = (
    df[df['primary_category'].isin(top5)]
    .groupby(['posting_month', 'primary_category'])
    .size()
    .reset_index(name='count')
)

fig = px.line(
    ind_mth, x='posting_month', y='count', color='primary_category',
    title='Monthly Postings — Top 5 Industries',
    labels={'posting_month': 'Month', 'count': 'Postings', 'primary_category': 'Industry'},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### EDA Key Findings

| # | Finding | Dashboard implication |
|---|---------|----------------------|
| 1 | **Admin/Secretarial, IT, Sales** are the top 3 industries by volume — together they account for >30% of all postings. | Make industry filter prominent. |
| 2 | **Permanent and Full Time** together cover ~81% of postings — part-time and contract are minority. | Employment-type filter less critical but still useful. |
| 3 | Median salary rises sharply from ~S$2,500 (entry) to ~S$8,000+ (Manager/Senior Management). | Salary-by-level chart is a core view. |
| 4 | ~55% of postings fall in the **Junior ($2.5k–$4.5k)** band — reflects a mid-level dominated market. | Salary band KPIs on overview. |
| 5 | **Posting volume is stable** mid-2023 onward at ~75k/month; no strong seasonality. | Line chart confirms steady demand. |
| 6 | Title normalisation collapses ≈15% of title variants; `supervisor` and `chef` have high case-variant noise. | Normalised titles give cleaner top-N ranking. |


---
## 7. Export Clean Dataset

Select the columns needed by the Streamlit dashboard and save as Parquet for fast loading.


In [28]:
KEEP_COLS = [
    'title', 'title_norm', 'primary_category',
    'employmentTypes', 'positionLevels', 'seniority',
    'salary_minimum', 'salary_maximum', 'average_salary', 'salary_band',
    'posting_date', 'posting_month', 'posting_year', 'is_repost',
    'postedCompany_name', 'status_jobStatus',
    'minimumYearsExperience', 'numberOfVacancies',
    'metadata_totalNumberJobApplication', 'metadata_totalNumberOfView',
]

df_out = df[KEEP_COLS].copy().reset_index(drop=True)
df_out.to_parquet('cleaned_jobs.parquet', index=False)

size_mb = os.path.getsize('cleaned_jobs.parquet') / 1_048_576
print(f'Saved {len(df_out):,} rows × {df_out.shape[1]} columns to cleaned_jobs.parquet ({size_mb:.1f} MB)')

Saved 1,036,744 rows × 20 columns to cleaned_jobs.parquet (55.2 MB)


In [29]:
# Sanity-check the exported file
verify = pd.read_parquet('cleaned_jobs.parquet')
print('Shape:', verify.shape)
print('dtypes:')
print(verify.dtypes.to_string())
print('\nSample:')
verify.head(3)

Shape: (1036744, 20)
dtypes:
title                                            str
title_norm                                       str
primary_category                                 str
employmentTypes                                  str
positionLevels                                   str
seniority                                        str
salary_minimum                                 int64
salary_maximum                                 int64
average_salary                               float64
salary_band                                      str
posting_date                          datetime64[us]
posting_month                                    str
posting_year                                   int32
is_repost                                       bool
postedCompany_name                               str
status_jobStatus                                 str
minimumYearsExperience                         int64
numberOfVacancies                              int64
metadata_totalNum

,title,title_norm,primary_category,employmentTypes,positionLevels,seniority,salary_minimum,salary_maximum,average_salary,salary_band,posting_date,posting_month,posting_year,is_repost,postedCompany_name,status_jobStatus,minimumYearsExperience,numberOfVacancies,metadata_totalNumberJobApplication,metadata_totalNumberOfView
0,Food Technologist - Clementi | Entry Level | U...,food technologist - clementi | entry level | u...,Environment / Health,Permanent,Executive,Mid,2000,2800,"2,400.00",Entry (<$2.5k),2023-03-30,2023-03,2023,True,WORKSTONE PTE. LTD.,Closed,0,1,5,151
1,"Software Engineer (Fab Support) (Java, CIM, Up...","software engineer (fab support) (java, cim, up...",Information Technology,Permanent,Executive,Mid,4000,5500,"4,750.00",Mid ($4.5k-$7.5k),2023-04-08,2023-04,2023,False,TRUST RECRUIT PTE. LTD.,Closed,2,2,0,55
2,Senior Technician,senior technician,Repair and Maintenance,Full Time,Senior Executive,Senior,3800,4600,"4,200.00",Junior ($2.5k-$4.5k),2023-04-08,2023-04,2023,False,PU TIEN SERVICES PTE. LTD.,Closed,3,1,7,99


---
## Next Steps

1. **Run the Streamlit dashboard:** `streamlit run streamlit_app.py`  
   The app reads `cleaned_jobs.parquet` directly — no DuckDB connection needed at runtime.
2. **Skill extraction** — parse job descriptions (if available) with keyword matching to surface in-demand skills.
3. **Role taxonomy** — cluster similar titles (e.g. NLP / fuzzy matching) to merge variants like *software engineer*, *software dev*, *SWE*.
4. **Predictive layer** — train a simple regressor on `positionLevels + primary_category + minimumYearsExperience` to predict salary band for new postings.
